In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import joblib


In [5]:
df = pd.read_csv("../data/cleaned_data.csv")
df.head()


,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
0,-1.033724,-0.108059,3,2.639545,4,3,2.351691,1.627173,2.648069,1,-0.730454,1
1,-1.223332,-1.661181,2,0.085307,1,1,-1.444385,0.041326,-0.672446,0,-0.999752,0
2,-0.464902,-1.661181,0,-0.990162,3,2,-0.672301,0.603522,2.648069,0,-0.730454,1
3,-0.844117,0.096299,3,-0.183561,3,2,2.351691,1.370448,2.648069,0,-0.999752,1
4,-0.654510,-0.252682,3,0.891908,3,2,2.351691,1.058478,2.648069,1,-0.461157,1


In [6]:
target_col = "loan_status"


In [7]:
print(df[target_col].value_counts())


loan_status
0    25327
1     7089
Name: count, dtype: int64


In [8]:
X = df.drop(columns=[target_col])
y = df[target_col]


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [10]:
print("Train Distribution:\n", y_train.value_counts())
print("Test Distribution:\n", y_test.value_counts())


Train Distribution:
 loan_status
0    20261
1     5671
Name: count, dtype: int64
Test Distribution:
 loan_status
0    5066
1    1418
Name: count, dtype: int64


In [11]:
def evaluate(y_true, y_pred, name):
    print("\nModel:", name)
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, zero_division=0))
    print("Recall:", recall_score(y_true, y_pred, zero_division=0))
    print("F1:", f1_score(y_true, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))


In [12]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
evaluate(y_test, y_pred_lr, "Logistic Regression")



Model: Logistic Regression
Accuracy: 0.8470080197409007
Precision: 0.725635593220339
Recall: 0.4830747531734838
F1: 0.5800169348010161
Confusion Matrix:
 [[4807  259]
 [ 733  685]]


In [13]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
evaluate(y_test, y_pred_lr, "Logistic Regression")



Model: Logistic Regression
Accuracy: 0.8470080197409007
Precision: 0.725635593220339
Recall: 0.4830747531734838
F1: 0.5800169348010161
Confusion Matrix:
 [[4807  259]
 [ 733  685]]


In [14]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
evaluate(y_test, y_pred_rf, "Random Forest")



Model: Random Forest
Accuracy: 0.9309068476249229
Precision: 0.9654510556621881
Recall: 0.7094499294781382
F1: 0.8178861788617886
Confusion Matrix:
 [[5030   36]
 [ 412 1006]]


In [15]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric="logloss")
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
evaluate(y_test, y_pred_xgb, "XGBoost")



Model: XGBoost
Accuracy: 0.938155459592844
Precision: 0.9721448467966574
Recall: 0.7383638928067701
F1: 0.8392785571142285
Confusion Matrix:
 [[5036   30]
 [ 371 1047]]


c:\Users\SAI\Desktop\FinGuard-AI\venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [21:47:49] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [16]:
scores = cross_val_score(rf, X, y, cv=5)
print("Cross-Validation Accuracy:", scores.mean())


Cross-Validation Accuracy: 0.9185279813583536


In [17]:
y_prob = rf.predict_proba(X_test)[:,1]
print("ROC AUC:", roc_auc_score(y_test, y_prob))


ROC AUC: 0.9318491678531675


In [18]:
joblib.dump(rf, "../models/credit_model.pkl")
print("Model Saved Successfully!")


Model Saved Successfully!


In [20]:
# ----- DASHBOARD EXPORT CELL -----

# make a copy so original df stays safe
df_dashboard = df.copy()

# create predictions using best model (rf)
df_dashboard["prediction"] = rf.predict(X)
df_dashboard["probability"] = rf.predict_proba(X)[:, 1]

# save CSV for Power BI / Tableau
df_dashboard.to_csv("../data/dashboard_data.csv", index=False)

print("Dashboard file created successfully!")


Dashboard file created successfully!
